In [11]:
import os
import sys
import platform
import json
import requests
import pandas as pd

from pathlib import Path
from huggingface_hub import HfApi

def resolve_project_dir() -> Path:
    starts: list[Path] = [Path.cwd().resolve()]
    for name in ("__vsc_ipynb_file__", "__file__"):
        if name in globals():
            starts.insert(0, Path(globals()[name]).resolve().parent)
    seen: set[Path] = set()
    for start in starts:
        for candidate in [start, *start.parents]:
            if candidate in seen:
                continue
            seen.add(candidate)
            if (candidate / "pyproject.toml").is_file():
                return candidate
            if (candidate / "data_access_env_audit.ipynb").is_file():
                return candidate
    return starts[0]


PROJECT_DIR = resolve_project_dir()
print("PROJECT_DIR:", PROJECT_DIR)

for folder in [
    "data/raw",
    "data/processed",
    "data/metadata",
    "outputs",
    "logs",
]:

    (PROJECT_DIR/folder).mkdir(parents=True, exist_ok=True)

PROJECT_DIR: /home/shrys/dev/Romcom-Drift


In [12]:
# auditing datasets, repo metadata
api = HfApi()

DATASETS = [
    "Divyanshu/indicxnli",
    "ai4bharat/IndicXParaphrase",
    "ai4bharat/indic-align",
]

dataset_inventory = []

for repo in DATASETS:

    print(f"\nInspecting: {repo}")

    try:
        info = api.dataset_info(
            repo_id=repo,
            files_metadata=False,
        )

        card = info.cardData or {}

        files = [
            file.rfilename
            for file in (info.siblings or [])
        ]

        result = {
            "repository": repo,
            "revision": info.sha,
            "license": card.get("license"),
            "number_of_files": len(files),
            "files": files[:15],
            "status": "ACCESSIBLE",
        }

        dataset_inventory.append(result)

        print(json.dumps(result, indent=2))

    except Exception as e:

        print("ACCESS ERROR:", str(e))

        dataset_inventory.append({
            "repository": repo,
            "status": "FAILED",
            "error": str(e),
        })


Inspecting: Divyanshu/indicxnli
{
  "repository": "Divyanshu/indicxnli",
  "revision": "7092c27872e919f31d0496fb8b9c47bd2cba3f6c",
  "license": [
    "cc0-1.0"
  ],
  "number_of_files": 59,
  "files": [
    ".gitattributes",
    ".gitignore",
    "README.md",
    "backward/dev/xnli_as.json",
    "backward/dev/xnli_bn.json",
    "backward/dev/xnli_gu.json",
    "backward/dev/xnli_hi.json",
    "backward/dev/xnli_kn.json",
    "backward/dev/xnli_ml.json",
    "backward/dev/xnli_mr.json",
    "backward/dev/xnli_or.json",
    "backward/dev/xnli_pa.json",
    "backward/dev/xnli_ta.json",
    "backward/dev/xnli_te.json",
    "backward/test/xnli_as.json"
  ],
  "status": "ACCESSIBLE"
}

Inspecting: ai4bharat/IndicXParaphrase
{
  "repository": "ai4bharat/IndicXParaphrase",
  "revision": "181031f9e69c465e0c72fc84fd33b66114adb3e8",
  "license": null,
  "number_of_files": 12,
  "files": [
    ".gitattributes",
    "IndicXParaphrase.py",
    "data/as.tsv",
    "data/bn.tsv",
    "data/gu.tsv",
  

In [13]:
DAKSHINA_URL = (
    "https://storage.googleapis.com/gresearch/dakshina/"
    "dakshina_dataset_v1.0.tar"
)

response = requests.head(
    DAKSHINA_URL,
    allow_redirects=True,
    timeout=30,
)

print("HTTP status:", response.status_code)

size = response.headers.get("Content-Length")

if size:
    print("Archive size:", round(int(size) / 1024**2, 2), "MB")
else:
    print("Archive size: Not reported")

print("Content type:", response.headers.get("Content-Type"))

HTTP status: 200
Archive size: 1915.3 MB
Content type: application/x-tar


In [14]:
audit_path = (
    PROJECT_DIR / "data/metadata/dataset_inventory.json"
)

audit_path.write_text(
    json.dumps(dataset_inventory, indent=2),
    encoding="utf-8",
)

print("Inventory saved:", audit_path)

Inventory saved: /home/shrys/dev/Romcom-Drift/data/metadata/dataset_inventory.json
